In [1]:
# %% [markdown]
# Notebook ETL – leitura em batch, validações e write-back


In [2]:
# %% [code]
import os
import sys
import logging
from dotenv import load_dotenv
from rich.console import Console
from rich.logging import RichHandler

# ── Ajuste do PYTHONPATH para permitir importar de logs/ ──────────────────
project_root = os.getcwd()  # supondo que o notebook esteja em /home/debrito/Documentos/etl_debrito
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from logs.logging_setup import get_logger  # importa o configurador unificado

# ── Carrega variáveis de ambiente de .env (opcional) ───────────────────────
load_dotenv()

# ── Configuração de logs no notebook ──────────────────────────────────────
console = Console(width=120)
root_logger = logging.getLogger()
root_logger.setLevel(logging.DEBUG)

# ── Atenção: NÃO remover handlers existentes (por exemplo, o FileHandler) ──
#for h in root_logger.handlers[:]:
#    root_logger.removeHandler(h)

# ── Cria apenas o RichHandler para console, mantendo o FileHandler intacto ─
rich_handler = RichHandler(
    console=console,
    rich_tracebacks=True,
    show_time=True,
    show_level=True,
    show_path=False,
    markup=True,
)
rich_handler.setLevel(logging.DEBUG)
rich_handler.setFormatter(
    logging.Formatter("%(asctime)s %(levelname)s %(name)s › %(message)s", datefmt="%H:%M:%S")
)
root_logger.addHandler(rich_handler)

# ── Logger específico para este notebook ──────────────────────────────────
log = get_logger(__name__)
log.debug("Logger configurado para o notebook (RichHandler + FileHandler ativos)")


17:38:38 DEBUG __main__ › Logger configurado para o notebook (RichHandler + FileHandler ativos)


17:38:38 DEBUG    17:38:38 DEBUG __main__ › Logger configurado para o notebook (RichHandler + FileHandler ativos)

In [3]:
#2 %% [code]
import math
import numpy as np
from typing import Any

def _to_json_safe(x: Any) -> Any:
    """
    Internal helper to convert arbitrary objects into JSON-safe primitives.
    """
    if x is None:
        return None
    if isinstance(x, (int, str, bool)):
        return x
    if isinstance(x, float):
        return None if math.isnan(x) or math.isinf(x) else x
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return None if (math.isnan(x) or math.isinf(x)) else float(x)
    if isinstance(x, (list, tuple, set)):
        return [_to_json_safe(item) for item in x]
    if isinstance(x, dict):
        return {k: _to_json_safe(v) for k, v in x.items()}
    # Fallback: stringify anything else
    return str(x)

def json_safe(obj: Any) -> Any:
    """
    Recursively converts `obj` into structures 100% serializable to JSON.
    """
    return _to_json_safe(obj)


In [4]:
#3 %% [code]
# Flags de gravação (ajuste conforme necessidade)
WRITE_BACK_ORIGIN = True   # grava na aba-origem (meta*, tiktok*, …)
WRITE_BACK_DEST   = True   # grava nas abas-modelo (modelo*)
DRY_RUN_DEST      = False  # True = simula write-back destino

# Credenciais e identificador da planilha
CREDS_PATH     = os.getenv("GOOGLE_CREDS_PATH", "creds.json")
SPREADSHEET_ID = os.getenv(
    "GOOGLE_SHEET_ID",
    "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"
)

# Abas de origem a processar, agrupadas por plataforma
SHEET_NAMES = [
    # Meta (Facebook / Instagram)
    "metaGeral",
    "metaIdade",
    "metaGenero",
    "metaRegiao",
    "metaAlcance",
    # TikTok
    "tiktokGeral",
    "tiktokIdade",
    "tiktokGenero",
    "tiktokRegiao",
    "tiktokAlcance",
    # Pinterest
    "pinterestGeral",
    "pinterestGenero",
    "pinterestIdade",
    "pinterestRegiao",
    "pinterestAlcance",
    # LinkedIn
    "linkedinGeral",
    "linkedinRegiao",
    "linkedinAlcance",
    # Google Analytics
    "GAGeral",
]


In [5]:
#4
# %% [code]
import importlib

# módulos principais e helpers recarregados
import extract.sheets_fetcher               as sf_mod
import treat.treat_pipeline                 as tp_mod
import treat.platforms                      as platforms_mod
import treat.platforms.linkedin             as linkedin_mod
import treat.platforms.tiktok               as tiktok_mod
import treat.platforms.pinterest            as pinterest_mod
import treat.platforms.meta                 as meta_mod
import treat.platforms.ga                   as ga_mod
import load.origin_writer                   as ow_mod
import load.dest_writer                     as dw_mod
import treat.utils.renomeacoes              as rn_mod
import treat.utils.preview_links            as prev_mod
import treat.utils.atribuicoes_via_lookup   as atrib_mod
import treat.utils.substitute_origin_values as sub_mod
import treat.utils.preprocess_utils         as pre_mod
import treat.utils.geo_normalize            as geo_mod

# hot-reload de todos os módulos alterados durante o desenvolvimento
for m in (
    sf_mod,
    tp_mod,
    platforms_mod,
    linkedin_mod,
    tiktok_mod,
    pinterest_mod,
    meta_mod,
    ga_mod,
    ow_mod,
    dw_mod,
    rn_mod,
    prev_mod,
    atrib_mod,
    sub_mod,
    pre_mod,
    geo_mod,
):
    importlib.reload(m)


In [6]:
# %% [code]
# Cell 5: definição do helper run_etl_for_sheet
import pandas as pd
import gc
import json
from pprint import pp
from typing import Dict

from logs.logging_setup import get_logger
log = get_logger(__name__)

from extract.sheets_fetcher import SheetsFetcher
from treat.treat_pipeline import TreatPipeline
from treat.utils.renomeacoes import renomeacao_geral, renomear_colunas_origem_para_modelo
from treat.utils.campos_calculados import calcular_engajamento_total, gerar_id
from load.origin_writer import write_back_origin
from load.dest_writer import write_back_for_sheet

# Instância única do fetcher (retry/backoff/cache interno)
fetcher = SheetsFetcher(
    spreadsheet_id=SPREADSHEET_ID,
    creds_path=CREDS_PATH,
)

def run_etl_for_sheet(
    *,
    sheet: str,
    wb_origin_flag: bool,
    wb_dest_flag: bool,
    dry_run_dest: bool,
    preloaded_raw: pd.DataFrame,
) -> Dict[str, pd.DataFrame | dict]:
    """
    Executa o fluxo completo para uma aba e devolve:
      { "dest": DataFrame destino (ou vazio), "taxo": relatório de taxonomia }
    """
    # 1) Dados brutos já carregados
    df_raw = preloaded_raw

    # 2) Tratamento via pipeline
    pipeline = TreatPipeline(
        creds_path=CREDS_PATH,
        spreadsheet_id=SPREADSHEET_ID,
        sheet_name=sheet,
        mapping_renomeacao=renomeacao_geral,
        write_back=wb_origin_flag,
    )
    df_ok = pipeline.run(df_raw)

    # 3) Relatório de taxonomia
    taxo_report = getattr(pipeline, "_last_taxo_report", {})
    pp(json.dumps(taxo_report, default=str), width=120)



    # 5) Preparar DataFrame de destino (modelo)
    df_model = renomear_colunas_origem_para_modelo(df_ok, renomeacao_geral)
    df_model = calcular_engajamento_total(df_model)
    df_model["ID"] = df_model.apply(gerar_id, axis=1)

    # 6) Write-back de destino  ──────────────────────────────────────────────
    if sheet.lower().startswith("ga"):
        log.info("🔸 %s: write-back de destino ignorado (Google Analytics)", sheet)
        df_dest = pd.DataFrame()      # retorna DataFrame vazio
    else:
        df_dest = write_back_for_sheet(
            df_model,
            sheet_name     = sheet,
            creds_path     = CREDS_PATH,
            spreadsheet_id = SPREADSHEET_ID,
            write_back     = wb_dest_flag,
            dry_run        = dry_run_dest,
        )
        if df_dest is None:
            df_dest = pd.DataFrame()

    # 7) Retorno
    return {"dest": df_dest, "taxo": taxo_report}


17:38:38 DEBUG urllib3.util.retry › Converted retries value: 3 -> Retry(total=3, connect=None, read=None, redirect=None, status=None)


         DEBUG    17:38:38 DEBUG urllib3.util.retry › Converted retries value: 3 -> Retry(total=3, connect=None,        
                  read=None, redirect=None, status=None)

In [7]:
# %% [code]
# Cell 6: Processamento em lote das abas (com logging via logs.logging_setup)
from contextlib import suppress
import gc
import pandas as pd
from tqdm.auto import tqdm

from logs.logging_setup import get_logger
log = get_logger(__name__)

from load.dest_writer import prefetch_meta

# 1) Leitura batch de todas as abas
all_raw = fetcher.get(SHEET_NAMES)

# 2) Copia cada DataFrame para não alterar in-place
all_raw = {name: df.copy() for name, df in all_raw.items()}

# 3) Registrar colunas originais de cada aba para debug
orig_columns_map = {name: df.columns.tolist() for name, df in all_raw.items()}
for name, cols in orig_columns_map.items():
    log.debug(f"Aba '{name}' colunas originais: {cols}")

# 4) Pré-busca de cabeçalhos e IDs das abas-modelo
prefetch_meta(fetcher, SPREADSHEET_ID)
log.info("📥 Prefetch meta concluído – começando processamento das abas")

# 5) Processamento aba a aba
results: dict[str, dict[str, object]] = {}

for sheet in tqdm(SHEET_NAMES, desc="Processando abas"):
    is_ga = sheet.lower().startswith("ga")
    if is_ga:
        log.info(f"🔸 {sheet}: apenas write-back de origem; destino será ignorado")

    out = run_etl_for_sheet(
        sheet           = sheet,
        wb_origin_flag  = WRITE_BACK_ORIGIN,
        wb_dest_flag    = WRITE_BACK_DEST,
        dry_run_dest    = DRY_RUN_DEST,
        preloaded_raw   = all_raw[sheet],
    )

    results[sheet] = {"dest": out["dest"], "taxo": out["taxo"]}
    log.debug(f"Aba '{sheet}' processada – resultados armazenados")

    gc.collect()

log.info("✅ Processamento de todas as abas concluído")


17:38:38 INFO extract.sheets_fetcher › 🔄 batchGet tentativa para ranges: ['metaGeral!A:ZZ', 'metaIdade!A:ZZ', 'metaGenero!A:ZZ', 'metaRegiao!A:ZZ', 'metaAlcance!A:ZZ', 'tiktokGeral!A:ZZ', 'tiktokIdade!A:ZZ', 'tiktokGenero!A:ZZ', 'tiktokRegiao!A:ZZ', 'tiktokAlcance!A:ZZ', 'pinterestGeral!A:ZZ', 'pinterestGenero!A:ZZ', 'pinterestIdade!A:ZZ', 'pinterestRegiao!A:ZZ', 'pinterestAlcance!A:ZZ', 'linkedinGeral!A:ZZ', 'linkedinRegiao!A:ZZ', 'linkedinAlcance!A:ZZ', 'GAGeral!A:ZZ']


         INFO     17:38:38 INFO extract.sheets_fetcher › 🔄 batchGet tentativa para ranges: ['metaGeral!A:ZZ',          
                  'metaIdade!A:ZZ', 'metaGenero!A:ZZ', 'metaRegiao!A:ZZ', 'metaAlcance!A:ZZ', 'tiktokGeral!A:ZZ',       
                  'tiktokIdade!A:ZZ', 'tiktokGenero!A:ZZ', 'tiktokRegiao!A:ZZ', 'tiktokAlcance!A:ZZ',                   
                  'pinterestGeral!A:ZZ', 'pinterestGenero!A:ZZ', 'pinterestIdade!A:ZZ', 'pinterestRegiao!A:ZZ',         
                  'pinterestAlcance!A:ZZ', 'linkedinGeral!A:ZZ', 'linkedinRegiao!A:ZZ', 'linkedinAlcance!A:ZZ',         
                  'GAGeral!A:ZZ']

17:38:38 DEBUG googleapiclient.discovery › URL being requested: GET https://sheets.googleapis.com/v4/spreadsheets/1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg/values:batchGet?ranges=metaGeral%21A%3AZZ&ranges=metaIdade%21A%3AZZ&ranges=metaGenero%21A%3AZZ&ranges=metaRegiao%21A%3AZZ&ranges=metaAlcance%21A%3AZZ&ranges=tiktokGeral%21A%3AZZ&ranges=tiktokIdade%21A%3AZZ&ranges=tiktokGenero%21A%3AZZ&ranges=tiktokRegiao%21A%3AZZ&ranges=tiktokAlcance%21A%3AZZ&ranges=pinterestGeral%21A%3AZZ&ranges=pinterestGenero%21A%3AZZ&ranges=pinterestIdade%21A%3AZZ&ranges=pinterestRegiao%21A%3AZZ&ranges=pinterestAlcance%21A%3AZZ&ranges=linkedinGeral%21A%3AZZ&ranges=linkedinRegiao%21A%3AZZ&ranges=linkedinAlcance%21A%3AZZ&ranges=GAGeral%21A%3AZZ&alt=json


         DEBUG    17:38:38 DEBUG googleapiclient.discovery › URL being requested: GET                                   
                  https://sheets.googleapis.com/v4/spreadsheets/1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg/values:batc
                  hGet?ranges=metaGeral%21A%3AZZ&ranges=metaIdade%21A%3AZZ&ranges=metaGenero%21A%3AZZ&ranges=metaRegiao%
                  21A%3AZZ&ranges=metaAlcance%21A%3AZZ&ranges=tiktokGeral%21A%3AZZ&ranges=tiktokIdade%21A%3AZZ&ranges=ti
                  ktokGenero%21A%3AZZ&ranges=tiktokRegiao%21A%3AZZ&ranges=tiktokAlcance%21A%3AZZ&ranges=pinterestGeral%2
                  1A%3AZZ&ranges=pinterestGenero%21A%3AZZ&ranges=pinterestIdade%21A%3AZZ&ranges=pinterestRegiao%21A%3AZZ
                  &ranges=pinterestAlcance%21A%3AZZ&ranges=linkedinGeral%21A%3AZZ&ranges=linkedinRegiao%21A%3AZZ&ranges=
                  linkedinAlcance%21A%3AZZ&ranges=GAGeral%21A%3AZZ&alt=json

17:38:38 DEBUG google_auth_httplib2 › Making request: POST https://oauth2.googleapis.com/token


...

         INFO     17:39:53 INFO __main__ › 🔸 GAGeral: write-back de destino ignorado (Google Analytics)

17:39:53 DEBUG __main__ › Aba 'GAGeral' processada – resultados armazenados


         DEBUG    17:39:53 DEBUG __main__ › Aba 'GAGeral' processada – resultados armazenados

17:39:53 INFO __main__ › ✅ Processamento de todas as abas concluído


         INFO     17:39:53 INFO __main__ › ✅ Processamento de todas as abas concluído

In [8]:
# %% [code]
# Cell 7: Validação de consistência de datas entre modelos e estatísticas de uso
from logs.logging_setup import get_logger
log = get_logger(__name__)

from treat.treat_pipeline import BIParamLookup
from treat.utils.validations import validate_consistent_dates_across_models

import gspread
import google.auth
from pprint import pprint

# ── 1) Extrair apenas os DataFrames de destino ─────────────────────────────
dest_dfs = {sheet: info["dest"] for sheet, info in results.items()}

# ── 2) Validar consistência de datas ───────────────────────────────────────
log.info("🔍 Validando consistência de datas entre modelos …")
df_inconsistencies = validate_consistent_dates_across_models(dest_dfs)

if df_inconsistencies is not None and not df_inconsistencies.empty:
    log.warning("💥 Inconsistências encontradas:")
    display(df_inconsistencies)
else:
    log.info("✅ Nenhuma divergência de start/end entre modelos.")

# ── 3) Limpar caches se necessário ─────────────────────────────────────────
# Limpa cache de leituras em lote (SheetsFetcher já instanciado como `fetcher`)
fetcher.refresh(SHEET_NAMES)
# Limpa cache da parametrização BI em memória
BIParamLookup._df = None
BIParamLookup._last_load = 0.0

# ── 4) Estatísticas de uso das planilhas ───────────────────────────────────
creds, _ = google.auth.load_credentials_from_file(CREDS_PATH, scopes=[
    "https://www.googleapis.com/auth/spreadsheets.readonly"
])
gc = gspread.authorize(creds)
ss = gc.open_by_key(SPREADSHEET_ID)

stats = []
for ws in ss.worksheets():
    rows = ws.row_count
    cols = ws.col_count
    cells = rows * cols
    stats.append((cells, ws.title, rows, cols))

stats.sort(reverse=True)          # maiores primeiro
log.info("📊 Top 10 abas que mais ocupam células:")
for cells, title, rows, cols in stats[:10]:
    log.info(f"  • {title}: {rows}×{cols} = {cells:,} células")


17:39:53 INFO __main__ › 🔍 Validando consistência de datas entre modelos …


         INFO     17:39:53 INFO __main__ › 🔍 Validando consistência de datas entre modelos …

17:39:53 WARNING treat.utils.validations › [Consistência Datas] Aba metaGeral: campanha='Empreendedorismo Feminino', veículo='Facebook' tem múltiplos start: {'2025-03-20', '2025-03-08'}


         WARNING  17:39:53 WARNING treat.utils.validations › [Consistência Datas] Aba metaGeral:                        
                  campanha='Empreendedorismo Feminino', veículo='Facebook' tem múltiplos start: {'2025-03-20',          
                  '2025-03-08'}

17:39:53 WARNING treat.utils.validations › [Consistência Datas] Aba metaGeral: campanha='Empreendedorismo Feminino', veículo='Instagram' tem múltiplos start: {'2025-03-20', '2025-03-08'}


...

         INFO     17:39:56 INFO __main__ ›   • SupermetricsQueries: 935×65 = 60,775 células

17:39:56 INFO __main__ ›   • modeloIdade: 4030×15 = 60,450 células


         INFO     17:39:56 INFO __main__ ›   • modeloIdade: 4030×15 = 60,450 células

17:39:56 INFO __main__ ›   • tiktokRegiao: 3024×15 = 45,360 células


         INFO     17:39:56 INFO __main__ ›   • tiktokRegiao: 3024×15 = 45,360 células

In [9]:
#8
# %% [code]
from treat.treat_pipeline import BIParamLookup

# — Limpa cache de leituras em lote (SheetsFetcher já instanciado como `fetcher` em células anteriores)
fetcher.refresh(SHEET_NAMES)

# — Limpa cache da parametrização BI em memória
BIParamLookup._df = None
BIParamLookup._last_load = 0.0


17:39:56 INFO extract.sheets_fetcher › 🔄 batchGet tentativa para ranges: ['metaGeral!A:ZZ', 'metaIdade!A:ZZ', 'metaGenero!A:ZZ', 'metaRegiao!A:ZZ', 'metaAlcance!A:ZZ', 'tiktokGeral!A:ZZ', 'tiktokIdade!A:ZZ', 'tiktokGenero!A:ZZ', 'tiktokRegiao!A:ZZ', 'tiktokAlcance!A:ZZ', 'pinterestGeral!A:ZZ', 'pinterestGenero!A:ZZ', 'pinterestIdade!A:ZZ', 'pinterestRegiao!A:ZZ', 'pinterestAlcance!A:ZZ', 'linkedinGeral!A:ZZ', 'linkedinRegiao!A:ZZ', 'linkedinAlcance!A:ZZ', 'GAGeral!A:ZZ']


         INFO     17:39:56 INFO extract.sheets_fetcher › 🔄 batchGet tentativa para ranges: ['metaGeral!A:ZZ',          
                  'metaIdade!A:ZZ', 'metaGenero!A:ZZ', 'metaRegiao!A:ZZ', 'metaAlcance!A:ZZ', 'tiktokGeral!A:ZZ',       
                  'tiktokIdade!A:ZZ', 'tiktokGenero!A:ZZ', 'tiktokRegiao!A:ZZ', 'tiktokAlcance!A:ZZ',                   
                  'pinterestGeral!A:ZZ', 'pinterestGenero!A:ZZ', 'pinterestIdade!A:ZZ', 'pinterestRegiao!A:ZZ',         
                  'pinterestAlcance!A:ZZ', 'linkedinGeral!A:ZZ', 'linkedinRegiao!A:ZZ', 'linkedinAlcance!A:ZZ',         
                  'GAGeral!A:ZZ']

17:39:56 DEBUG googleapiclient.discovery › URL being requested: GET https://sheets.googleapis.com/v4/spreadsheets/1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg/values:batchGet?ranges=metaGeral%21A%3AZZ&ranges=metaIdade%21A%3AZZ&ranges=metaGenero%21A%3AZZ&ranges=metaRegiao%21A%3AZZ&ranges=metaAlcance%21A%3AZZ&ranges=tiktokGeral%21A%3AZZ&ranges=tiktokIdade%21A%3AZZ&ranges=tiktokGenero%21A%3AZZ&ranges=tiktokRegiao%21A%3AZZ&ranges=tiktokAlcance%21A%3AZZ&ranges=pinterestGeral%21A%3AZZ&ranges=pinterestGenero%21A%3AZZ&ranges=pinterestIdade%21A%3AZZ&ranges=pinterestRegiao%21A%3AZZ&ranges=pinterestAlcance%21A%3AZZ&ranges=linkedinGeral%21A%3AZZ&ranges=linkedinRegiao%21A%3AZZ&ranges=linkedinAlcance%21A%3AZZ&ranges=GAGeral%21A%3AZZ&alt=json


         DEBUG    17:39:56 DEBUG googleapiclient.discovery › URL being requested: GET                                   
                  https://sheets.googleapis.com/v4/spreadsheets/1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg/values:batc
                  hGet?ranges=metaGeral%21A%3AZZ&ranges=metaIdade%21A%3AZZ&ranges=metaGenero%21A%3AZZ&ranges=metaRegiao%
                  21A%3AZZ&ranges=metaAlcance%21A%3AZZ&ranges=tiktokGeral%21A%3AZZ&ranges=tiktokIdade%21A%3AZZ&ranges=ti
                  ktokGenero%21A%3AZZ&ranges=tiktokRegiao%21A%3AZZ&ranges=tiktokAlcance%21A%3AZZ&ranges=pinterestGeral%2
                  1A%3AZZ&ranges=pinterestGenero%21A%3AZZ&ranges=pinterestIdade%21A%3AZZ&ranges=pinterestRegiao%21A%3AZZ
                  &ranges=pinterestAlcance%21A%3AZZ&ranges=linkedinGeral%21A%3AZZ&ranges=linkedinRegiao%21A%3AZZ&ranges=
                  linkedinAlcance%21A%3AZZ&ranges=GAGeral%21A%3AZZ&alt=json

17:39:58 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGeral!A1:AB588


...

         INFO     17:39:58 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: linkedinAlcance!A1:J2

17:39:58 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: GAGeral!A1:X4693


         INFO     17:39:58 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: GAGeral!A1:X4693

17:39:58 INFO extract.sheets_fetcher › 📡 batchGet 19 ranges


         INFO     17:39:58 INFO extract.sheets_fetcher › 📡 batchGet 19 ranges

In [10]:
#9
# — Forçar recarregamento dos parâmetros BI em qualquer ponto do notebook —
from treat.bi_param_utils import BIParamLookup

# Zera o cache interno para que a próxima chamada a .df() refaça o carregamento
BIParamLookup._df = None
BIParamLookup._last_load = 0.0


In [11]:
#10 – Validar consistência de datas entre modelos

from treat.utils.validations import validate_consistent_dates_across_models

# Extrai apenas os DataFrames de destino
dest_dfs = {sheet: info["dest"] for sheet, info in results.items()}

# Roda a validação
df_inconsistencies = validate_consistent_dates_across_models(dest_dfs)

# Exibe resultados
if df_inconsistencies is not None and not df_inconsistencies.empty:
    display(df_inconsistencies)
else:
    print("✅ Nenhuma divergência de start/end entre modelos.")


17:39:58 WARNING treat.utils.validations › [Consistência Datas] Aba metaGeral: campanha='Empreendedorismo Feminino', veículo='Facebook' tem múltiplos start: {'2025-03-20', '2025-03-08'}


         WARNING  17:39:58 WARNING treat.utils.validations › [Consistência Datas] Aba metaGeral:                        
                  campanha='Empreendedorismo Feminino', veículo='Facebook' tem múltiplos start: {'2025-03-20',          
                  '2025-03-08'}

17:39:58 WARNING treat.utils.validations › [Consistência Datas] Aba metaGeral: campanha='Empreendedorismo Feminino', veículo='Instagram' tem múltiplos start: {'2025-03-20', '2025-03-08'}


         WARNING  17:39:58 WARNING treat.utils.validations › [Consistência Datas] Aba metaGeral:                        
                  campanha='Empreendedorismo Feminino', veículo='Instagram' tem múltiplos start: {'2025-03-20',         
                  '2025-03-08'}

17:39:58 WARNING treat.utils.validations › [Consistência Datas] Aba metaGeral: campanha='Inova Cerrado e Pantanal', veículo='Facebook' tem múltiplos start: {'2025-03-12', '2025-03-13'}


...

17:39:58 WARNING treat.utils.validations › [Consistência Datas] Entre abas: campanha='Inova Cerrado e Pantanal', veículo='Facebook' tem inconsistência start/end em metaGeral


         WARNING  17:39:58 WARNING treat.utils.validations › [Consistência Datas] Entre abas: campanha='Inova Cerrado e 
                  Pantanal', veículo='Facebook' tem inconsistência start/end em metaGeral

17:39:58 WARNING treat.utils.validations › [Consistência Datas] Entre abas: campanha='Empreendedorismo Feminino', veículo='Facebook' tem inconsistência start/end em metaGeral


         WARNING  17:39:58 WARNING treat.utils.validations › [Consistência Datas] Entre abas: campanha='Empreendedorismo
                  Feminino', veículo='Facebook' tem inconsistência start/end em metaGeral

,nível,aba,Campanha,Veiculo,start,end
0,aba,metaGeral,Empreendedorismo Feminino,Facebook,"{2025-03-20, 2025-03-08}",{}
1,aba,metaGeral,Empreendedorismo Feminino,Instagram,"{2025-03-20, 2025-03-08}",{}
2,aba,metaGeral,Inova Cerrado e Pantanal,Facebook,"{2025-03-12, 2025-03-13}",{}
3,aba,metaGeral,Inova Pantanal,Facebook,"{2025-03-12, 2025-03-13}",{}
4,aba,metaGeral,Inova Cerrado e Pantanal,Instagram,"{2025-03-12, 2025-03-13}",{}
5,aba,metaGeral,Inova Pantanal,Instagram,"{2025-03-12, 2025-03-13}",{}
6,aba,tiktokGeral,Empreendedorismo Feminino,TikTok,"{2025-03-21, 2025-03-08}",{}
7,aba,pinterestGeral,Empreendedorismo Feminino,Pinterest,"{2025-03-20, 2025-03-07, 2025-03-08}",{}
8,aba,tiktokGeral,Empreendedorismo Feminino,TikTok,{},"{2025-04-01, 2025-03-31, 2025-04-04}"
9,entre-abas,metaGeral,Empreendedorismo Feminino,Instagram,"{2025-03-20, 2025-03-08}",{2025-03-31}


In [12]:
import gspread, google.auth
from pprint import pprint

creds, _ = google.auth.load_credentials_from_file(CREDS_PATH, scopes=[
    "https://www.googleapis.com/auth/spreadsheets.readonly"
])
gc = gspread.authorize(creds)
ss = gc.open_by_key(SPREADSHEET_ID)

stats = []
for ws in ss.worksheets():
    rows = ws.row_count
    cols = ws.col_count
    cells = rows * cols
    stats.append((cells, ws.title, rows, cols))

stats.sort(reverse=True)          # maiores primeiro
pprint(stats[:40])                # top 10 abas que mais ocupam células


17:39:58 DEBUG urllib3.util.retry › Converted retries value: 3 -> Retry(total=3, connect=None, read=None, redirect=None, status=None)


         DEBUG    17:39:58 DEBUG urllib3.util.retry › Converted retries value: 3 -> Retry(total=3, connect=None,        
                  read=None, redirect=None, status=None)

17:39:58 DEBUG urllib3.connectionpool › Starting new HTTPS connection (1): oauth2.googleapis.com:443


         DEBUG    17:39:58 DEBUG urllib3.connectionpool › Starting new HTTPS connection (1): oauth2.googleapis.com:443

17:39:58 DEBUG urllib3.connectionpool › https://oauth2.googleapis.com:443 "POST /token HTTP/1.1" 200 None


...

17:39:59 DEBUG urllib3.connectionpool › https://sheets.googleapis.com:443 "GET /v4/spreadsheets/1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg?includeGridData=false HTTP/1.1" 200 None


17:39:59 DEBUG    17:39:59 DEBUG urllib3.connectionpool › https://sheets.googleapis.com:443 "GET                        
                  /v4/spreadsheets/1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg?includeGridData=false HTTP/1.1" 200 None

17:39:59 DEBUG urllib3.connectionpool › https://sheets.googleapis.com:443 "GET /v4/spreadsheets/1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg?includeGridData=false HTTP/1.1" 200 None


         DEBUG    17:39:59 DEBUG urllib3.connectionpool › https://sheets.googleapis.com:443 "GET                        
                  /v4/spreadsheets/1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg?includeGridData=false HTTP/1.1" 200 None

[(786615, 'modeloGenero', 52441, 15),
 (525270, 'modeloRegiao', 35018, 15),
 (300510, 'LINKEDIN NEGOCIOS ALCANÇADOS', 11130, 27),
 (192588, 'metaPontoControle', 16049, 12),
 (170000, 'metaRegiao', 10000, 17),
 (122018, 'modeloGeral', 4693, 26),
 (112632, 'GAGeral', 4693, 24),
 (60775, 'SupermetricsQueries', 935, 65),
 (60450, 'modeloIdade', 4030, 15),
 (45360, 'tiktokRegiao', 3024, 15),
 (44400, 'modeloAlcance', 3700, 12),
 (36380, 'metaIdade', 2140, 17),
 (36000, 'kawaiGeral', 1000, 36),
 (32788, 'CONTEÚDO _MÍDIA', 1171, 28),
 (28000, 'LEGENDA', 1000, 28),
 (26000, 'sites', 1000, 26),
 (26000, 'googleRegiao', 1000, 26),
 (26000, 'googleIdade', 1000, 26),
 (26000, 'googleGenero', 1000, 26),
 (26000, 'cidade', 1000, 26),
 (26000, 'adserver', 1000, 26),
 (26000, 'MEDIUM', 1000, 26),
 (26000, 'GAEventos', 1000, 26),
 (26000, 'CATALISA', 1000, 26),
 (26000, 'Alright genero', 1000, 26),
 (26000, 'Alright dispositivo', 1000, 26),
 (26000, 'Alright Impressões diária', 1000, 26),
 (26000, 'Alr